In [3]:
# 目标：对 ./data/metadata_merged_raw.csv 做与之前相同的清洗与特征处理，
#     并导出为 ./data/metadata_merged.csv；
#     写出时仅“文本列”加双引号，日期与数值不加引号。

import pandas as pd
import numpy as np

# === 1) 读取原始融合文件 ===
df = pd.read_csv("../data/metadata_merged_raw.csv", low_memory=False)

# === 2) 列顺序整理：确保 Title/Slug 为第2/3列，Tags 为第4列；随后按 Id 升序 ===
cols = df.columns.tolist()
reordered = ["Id", "Title", "Slug", "Tags"] + [c for c in cols if c not in ["Id","Title","Slug","Tags"]]
df = df[reordered]
df = df.sort_values("Id", ascending=True).reset_index(drop=True)

# === 3) 解析日期列为 datetime（便于后续衍生时间特征） ===
df["CreationDate_dt"] = pd.to_datetime(df["CreationDate"], errors="coerce")
df["LastActivityDate_dt"] = pd.to_datetime(df["LastActivityDate"], errors="coerce")

# === 4) 派生时间特征：年龄/最近活跃天数/近30天是否活跃 ===
today = pd.Timestamp.today().normalize()
df["age_days"] = (today - df["CreationDate_dt"]).dt.days
df["days_since_last_activity"] = (today - df["LastActivityDate_dt"]).dt.days
df["active_30d"] = df["days_since_last_activity"].le(30).astype("boolean")

# === 5) 类型纠正：应为整数却因缺失被读成 float 的列 → 可空整型 Int64 ===
for c in ["OwnerUserId", "OwnerOrganizationId", "CurrentDatasetVersionId", "CurrentDatasourceVersionId", "Medal"]:
    if c in df.columns:
        df[c] = df[c].astype("Int64")

# === 6) 标签列修正：缺失填空串，并提供 has_tags 标志位 ===
df["Tags"] = df["Tags"].fillna("")
df["has_tags"] = df["Tags"].str.len().gt(0)

# === 7) 长尾计数列稳定化：log1p 特征（保留原列，新增 *_log1p） ===
for c in ["TotalViews", "TotalDownloads", "TotalVotes", "TotalKernels"]:
    if c in df.columns:
        df[c] = df[c].clip(lower=0)
        df[c + "_log1p"] = np.log1p(df[c])

# === 8) 仅给“文本列”加双引号：自动识别日期样式列（≥90% 可解析为日期的不加引号） ===
obj_cols = [c for c in df.columns if df[c].dtype == "object"]
date_like = []
for c in obj_cols:
    parsed = pd.to_datetime(df[c], errors="coerce")
    if parsed.notna().mean() >= 0.90:   # 90% 以上可被解析为日期 → 视为日期列
        date_like.append(c)
text_cols = [c for c in obj_cols if c not in date_like]

# === 9) 手写 CSV：文本列强制双引号，日期/数值列不加引号；保存为 metadata_merged.csv ===
out_path = "../data/metadata_merged.csv"
with open(out_path, "w", encoding="utf-8", newline="") as f:
    f.write(",".join(df.columns) + "\n")  # 表头不加引号
    for row in df.itertuples(index=False, name=None):
        fields = []
        for col, val in zip(df.columns, row):
            if col in text_cols:
                if pd.isna(val):
                    fields.append('""')  # 文本NA写成空引号
                else:
                    s = str(val).replace('"', '""')  # 内部引号转义为两连引号
                    fields.append(f'"{s}"')
            else:
                fields.append("" if pd.isna(val) else str(val))  # 日期/数值：不加引号
        f.write(",".join(fields) + "\n")

print("saved:", out_path, "shape:", df.shape)
print("text-quoted columns:", text_cols)
print("date-like (unquoted) object columns:", date_like)


/var/folders/g3/l60s63pn30b_hr070pv7jw_h0000gn/T/ipykernel_76876/4019667042.py:46: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(df[c], errors="coerce")
/var/folders/g3/l60s63pn30b_hr070pv7jw_h0000gn/T/ipykernel_76876/4019667042.py:46: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  parsed = pd.to_datetime(df[c], errors="coerce")
/var/folders/g3/l60s63pn30b_hr070pv7jw_h0000gn/T/ipykernel_76876/4019667042.py:46: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and a

saved: ../data/metadata_merged.csv shape: (521735, 31)
text-quoted columns: ['Title', 'Slug', 'Tags', 'Type', 'MedalAwardDate', 'Subtitle', 'Description']
date-like (unquoted) object columns: ['CreationDate', 'LastActivityDate']
